In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

In [3]:
# Show every single row
pd.set_option('display.max_rows', None)

# Show every single column
pd.set_option('display.max_columns', None)

# Prevent long strings from being cut off in the middle
pd.set_option('display.max_colwidth', None)

np.set_printoptions(threshold=np.inf)

In [4]:
file_path = "final_ball_by_ball_first_innings.csv"  
df = pd.read_csv(file_path)

In [14]:
df = df.sort_values(by=["match_id", "over", "ball"]).reset_index(drop=True)

for c in df.columns:
    print(c, " ", df[c].nunique())

match_id   4913
date   2096
venue   446
innings   1
team   107
over   20
ball   19
batter   5582
bowler   4045
non_striker   5398
runs_batter   8
runs_extras   6
runs_total   8
wicket_fallen   2


In [9]:
df['team'].unique().tolist()

['England',
 'Australia',
 'South Africa',
 'Sri Lanka',
 'West Indies',
 'New Zealand',
 'Kenya',
 'Pakistan',
 'India',
 'Bangladesh',
 'Bermuda',
 'Scotland',
 'Ireland',
 'Zimbabwe',
 'Canada',
 'Netherlands',
 'Nepal',
 'United Arab Emirates',
 'Hong Kong',
 'Oman',
 'Papua New Guinea',
 'Thailand',
 'Uganda',
 'Malaysia',
 'Botswana',
 'Malawi',
 'Sierra Leone',
 'Mozambique',
 'Namibia',
 'Lesotho',
 'China',
 'Kuwait',
 'Vanuatu',
 'Philippines',
 'United States of America',
 'Germany',
 'Nigeria',
 'Tanzania',
 'Japan',
 'Indonesia',
 'Fiji',
 'Samoa',
 'Ghana',
 'Guernsey',
 'Denmark',
 'Jersey',
 'Italy',
 'Norway',
 'Maldives',
 'Mali',
 'Singapore',
 'Cayman Islands',
 'Portugal',
 'Gibraltar',
 'Spain',
 'Bhutan',
 'Qatar',
 'Iran',
 'Austria',
 'Belgium',
 'Isle of Man',
 'Bulgaria',
 'Romania',
 'Luxembourg',
 'Czech Republic',
 'Rwanda',
 'Greece',
 'Serbia',
 'Malta',
 'France',
 'Sweden',
 'Finland',
 'Eswatini',
 'Cameroon',
 'Hungary',
 'Estonia',
 'Cyprus',
 'Swit

In [19]:
batters = df['batter'].unique().tolist()
bowlers = df['bowler'].unique().tolist()
players = np.union1d(batters, bowlers).tolist()
len(players)

6231

In [22]:
batters_not_bowlers = np.setdiff1d(batters, bowlers).tolist()
len(batters_not_bowlers)

2186

In [23]:
bowlers_not_batters = np.setdiff1d(bowlers, batters).tolist()
len(bowlers_not_batters)

649

In [24]:
batter_and_bowler = np.intersect1d(batters, bowlers).tolist()
len(batter_and_bowler)

3396

In [34]:
nonstrikers = df['non_striker'].unique().tolist()
nonstriker_notbatter_notbowler = np.setdiff1d(nonstrikers, batters + bowlers).tolist()
len(nonstriker_notbatter_notbowler)

21

In [31]:
venues = df['venue'].unique().tolist()
np.sort(venues)

array(['7he Sevens Stadium, Dubai', 'AMI Stadium',
       'Achimota Senior Secondary School A Field, Accra',
       'Achimota Senior Secondary School B Field, Accra', 'Adelaide Oval',
       'Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1)',
       'Al Amerat Cricket Ground Oman Cricket (Ministry Turf 2)',
       'Albert Park 1, Suva', 'Albert Park 2, Suva',
       'Albertslund Cricket Club', 'Allan Border Field',
       'Allan Border Field, Brisbane', 'Amini Park',
       'Amini Park, Port Moresby', 'Arnos Vale Ground, Kingstown',
       'Arnos Vale Ground, Kingstown, St Vincent', 'Arun Jaitley Stadium',
       'Arun Jaitley Stadium, Delhi',
       'Arundel Castle Cricket Club Ground',
       'Asian Institute of Technology Ground',
       'Asian Institute of Technology Ground, Bangkok',
       'Ballpark Ground, Graz', 'Barabati Stadium',
       'Barabati Stadium, Cuttack', 'Barsapara Cricket Stadium',
       'Barsapara Cricket Stadium, Guwahati', 'Basin Reserve',
       'Basin

In [ ]:

# ===========================================================
# 3. FEATURE ENGINEERING
# ===========================================================
df["runs_so_far"] = df.groupby("match_id")["runs_total"].cumsum()
df["wickets_so_far"] = df.groupby("match_id")["wicket_fallen"].cumsum()

df["balls_so_far"] = df.groupby("match_id").cumcount() + 1
df["overs_so_far"] = df["balls_so_far"] / 6.0

df["current_run_rate"] = df["runs_so_far"] / df["overs_so_far"].replace(0, np.nan)

df["balls_remaining"] = 120 - df["balls_so_far"]
df["wickets_remaining"] = 10 - df["wickets_so_far"]

# ===========================================================
# 4. FINAL SCORE PER MATCH
# ===========================================================
final_score = df.groupby("match_id")["runs_total"].sum().reset_index()
final_score = final_score.rename(columns={"runs_total": "final_score"})
df = df.merge(final_score, on="match_id")